# 04. Estadística inferencial

Contraste de las diferencias observadas en el notebook 03. Se usan tests no paramétricos porque `ArrDelay` no sigue una distribución normal (asimetría y cola extrema). En cada test se reporta el tamaño del efecto, no solo el p-valor, dado el gran tamaño muestral. Las comparaciones múltiples se corrigen con Benjamini-Hochberg (FDR).

### 1. Retraso y clima adverso (Mann-Whitney U)

Se contrasta si la distribución de `ArrDelay` difiere entre vuelos con y sin clima adverso, y entre vuelos con y sin nieve aproximada. Como tamaño del efecto se usa r = Z / √n (Rosenthal), interpretado como pequeño (~0,1), mediano (~0,3) o grande (~0,5).

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_parquet("../data/processed/vuelos_clima_2025q1.parquet")
df["estado"] = df["estado"].astype("category")
normal = df[df["estado"] == "normal"]

def mannwhitney_efecto(grupo_a, grupo_b, nombre):
    n1, n2 = len(grupo_a), len(grupo_b)
    u, p = stats.mannwhitneyu(grupo_a, grupo_b, alternative="two-sided")
    n = n1 + n2

    # z calculada directamente desde u
    media_u = n1 * n2 / 2
    var_u = n1 * n2 * (n + 1) / 12
    z = (u - media_u) / np.sqrt(var_u)
    r = abs(z) / np.sqrt(n)

    print(f"{nombre}: U={u:.0f}, p={p:.2e}, z={z:.1f}, r={r:.3f}, "
          f"medianas: {grupo_a.median():.0f} vs {grupo_b.median():.0f} min")
    return p, r

resultados = []
for col, etiqueta in [("clima_adverso", "Clima adverso"), ("nieve_aprox", "Nieve aproximada")]:
    con = normal.loc[normal[col], "ArrDelay"].dropna()
    sin = normal.loc[~normal[col], "ArrDelay"].dropna()
    p, r = mannwhitney_efecto(con, sin, etiqueta)
    resultados.append({"test": etiqueta, "p_valor": p, "efecto_r": r})

Clima adverso: U=89662340235, p=0.00e+00, z=114.1, r=0.108, medianas: -1 vs -8 min
Nieve aproximada: U=30698550228, p=0.00e+00, z=111.2, r=0.105, medianas: 9 vs -8 min


Ambas diferencias son estadísticamente significativas (p < 0,001 en los dos casos), lo cual es esperable con más de un millón de vuelos: incluso diferencias pequeñas son detectables. Por eso interesa más el tamaño del efecto: r = 0,108 para clima adverso y r = 0,105 para nieve aproximada, ambos en el rango de efecto **pequeño** (< 0,3 según Rosenthal), pese a que la diferencia de medianas es de 7 y 17 min respectivamente.

**Interpretación.** El clima adverso y la nieve se asocian de forma consistente con más retraso, pero explican solo una parte modesta de la variabilidad total de `ArrDelay`. Esto es coherente con lo que sugería la correlación de Spearman del notebook 03 (0,137 para precipitación): el retraso depende de muchos factores además del clima de origen (congestión de red, aerolínea, franja horaria), así que el efecto climático, aunque real, no domina la variabilidad global.

**Nota metodológica.** El cálculo de r usa la aproximación normal sin corrección por empates (`ArrDelay` en minutos enteros genera muchos valores repetidos); con esta muestra el efecto de la corrección es despreciable.

## 2. Correlación de Spearman entre clima y retraso

Se calcula el coeficiente de Spearman entre las variables continuas de clima y `ArrDelay`, con su p-valor, y se corrige por comparaciones múltiples (Benjamini-Hochberg), dado que se prueban varias variables a la vez.

In [3]:
from statsmodels.stats.multitest import multipletests

clima_cols = ["prcp_orig", "tavg_orig", "wspd_orig", "pres_orig"]
filas = []
for col in clima_cols:
    valido = normal[[col, "ArrDelay"]].dropna()
    rho, p = stats.spearmanr(valido[col], valido["ArrDelay"])
    filas.append({"variable": col, "rho": rho, "p_valor": p, "n": len(valido)})

tabla = pd.DataFrame(filas)
tabla["p_ajustado"] = multipletests(tabla["p_valor"], method="fdr_bh")[1]
tabla["significativo"] = tabla["p_ajustado"] < 0.05
print(tabla.round(4))

    variable     rho  p_valor        n  p_ajustado  significativo
0  prcp_orig  0.1372      0.0  1116207         0.0           True
1  tavg_orig -0.0053      0.0  1116207         0.0           True
2  wspd_orig  0.0693      0.0  1116207         0.0           True
3  pres_orig -0.0289      0.0  1116207         0.0           True


Las cuatro variables son estadísticamente significativas tras la corrección por comparaciones múltiples (p_ajustado < 0,001), de nuevo por el gran tamaño muestral (más de 1,1 millones de pares). En magnitud, todas son débiles: precipitación (ρ = 0,137) y viento (ρ = 0,069) muestran una asociación positiva con el retraso, coherente con lo ya visto en el análisis descriptivo. La presión atmosférica tiene una asociación negativa débil (ρ = -0,029): a menor presión (asociada a sistemas de baja presión y mal tiempo), algo más de retraso. La temperatura prácticamente no se asocia con el retraso (ρ = -0,005), pese a ser significativa.

El p-valor exacto no es representable (por debajo de la precisión de punto flotante) dado el tamaño muestral, por lo que el ajuste por FDR no cambia ninguna conclusión aquí. La variable con mayor peso es la precipitación, seguida del viento; la temperatura no aporta información relevante sobre el retraso y no se recomienda destacarla en el informe.

## 3. Diferencias de retraso por aerolínea, franja horaria y día de la semana (Kruskal-Wallis)

Se contrasta si `ArrDelay` difiere entre los grupos de cada variable categórica (más de dos grupos). Como tamaño del efecto se usa épsilon al cuadrado (ε² = (H - k + 1) / (n - k)), interpretado como pequeño (~0,01), mediano (~0,06) o grande (~0,14).

In [4]:
def kruskal_efecto(datos, col_grupo, col_valor, nombre):
    grupos = [g[col_valor].dropna().values for _, g in datos.groupby(col_grupo, observed=True)]
    h, p = stats.kruskal(*grupos)
    n = sum(len(g) for g in grupos)
    k = len(grupos)
    eps2 = (h - k + 1) / (n - k)
    print(f"{nombre}: H={h:.1f}, gl={k-1}, p={p:.2e}, ε²={eps2:.4f}, n_grupos={k}")
    return {"variable": nombre, "H": h, "p_valor": p, "epsilon2": eps2}

filas = []
for col, etiqueta in [("Reporting_Airline", "Aerolínea"), ("franja_hora", "Franja horaria"), ("dia_semana", "Día de la semana")]:
    if col == "dia_semana" and col not in normal.columns:
        dias = {1: "Lunes", 2: "Martes", 3: "Miércoles", 4: "Jueves", 5: "Viernes", 6: "Sábado", 7: "Domingo"}
        normal = normal.assign(dia_semana=normal["DayOfWeek"].map(dias))
    filas.append(kruskal_efecto(normal, col, "ArrDelay", etiqueta))

tabla_kw = pd.DataFrame(filas)
tabla_kw["p_ajustado"] = multipletests(tabla_kw["p_valor"], method="fdr_bh")[1]
print("\n", tabla_kw.round(4))

Aerolínea: H=7279.7, gl=13, p=0.00e+00, ε²=0.0065, n_grupos=14
Franja horaria: H=17968.9, gl=3, p=0.00e+00, ε²=0.0161, n_grupos=4
Día de la semana: H=10679.1, gl=6, p=0.00e+00, ε²=0.0096, n_grupos=7

            variable           H  p_valor  epsilon2  p_ajustado
0         Aerolínea   7279.6640      0.0    0.0065         0.0
1    Franja horaria  17968.9062      0.0    0.0161         0.0
2  Día de la semana  10679.1237      0.0    0.0096         0.0


Las tres variables muestran diferencias significativas en `ArrDelay` (p < 0,001 tras el ajuste FDR), de nuevo esperable con más de un millón de vuelos. En tamaño del efecto, las tres son pequeñas (ε² < 0,06): franja horaria es la más influyente (ε² = 0,0161), seguida de día de la semana (0,0096) y aerolínea (0,0065), la más débil de las tres pese a tener el mayor estadístico H, que aquí refleja el número de grupos (14) más que la magnitud real de la diferencia.

**Interpretación.** El orden de importancia coincide con lo observado en el análisis descriptivo: la franja horaria es el factor con mayor peso relativo, aunque ninguna de las tres variables por sí sola explica una parte sustancial de la variabilidad del retraso.

## 4. Cancelación y clima adverso / aerolínea (chi-cuadrado)

Se contrasta si la cancelación (variable binaria) está asociada con el clima adverso y con la aerolínea, mediante tablas de contingencia. Como tamaño del efecto se usa V de Cramér, interpretado como pequeño (~0,1), mediano (~0,3) o grande (~0,5) para tablas 2×2; para más de 2 grupos el umbral se ajusta por los grados de libertad.

In [5]:
def chi2_efecto(tabla, nombre):
    chi2, p, gl, esperado = stats.chi2_contingency(tabla)
    n = tabla.values.sum()
    k = min(tabla.shape) - 1
    v = np.sqrt(chi2 / (n * k))
    print(f"{nombre}: chi2={chi2:.1f}, gl={gl}, p={p:.2e}, V de Cramér={v:.4f}, n={n}")
    return {"variable": nombre, "chi2": chi2, "p_valor": p, "V_cramer": v}

filas = []

tabla_clima = pd.crosstab(df["clima_adverso"], df["estado"] == "cancelado")
filas.append(chi2_efecto(tabla_clima, "Clima adverso vs cancelación"))

tabla_aerolinea = pd.crosstab(df["Reporting_Airline"], df["estado"] == "cancelado")
filas.append(chi2_efecto(tabla_aerolinea, "Aerolínea vs cancelación"))

tabla_chi2 = pd.DataFrame(filas)
tabla_chi2["p_ajustado"] = multipletests(tabla_chi2["p_valor"], method="fdr_bh")[1]
print("\n", tabla_chi2.round(4))

Clima adverso vs cancelación: chi2=11634.2, gl=1, p=0.00e+00, V de Cramér=0.1011, n=1138858
Aerolínea vs cancelación: chi2=9290.6, gl=13, p=0.00e+00, V de Cramér=0.0903, n=1138858

                        variable        chi2  p_valor  V_cramer  p_ajustado
0  Clima adverso vs cancelación  11634.1944      0.0    0.1011         0.0
1      Aerolínea vs cancelación   9290.6413      0.0    0.0903         0.0


Ambas asociaciones son significativas (p < 0,001 tras FDR). El tamaño del efecto es similar en las dos: V = 0,101 para clima adverso y V = 0,090 para aerolínea, en la frontera entre pequeño y moderado, algo mayor que los efectos de Kruskal-Wallis sobre `ArrDelay`. Esto sugiere que el clima adverso y la aerolínea influyen algo más en si un vuelo se cancela que en cuánto se retrasa si vuela.

## 5. Resumen de la estadística inferencial

| Test | Comparación | p_ajustado | Tamaño del efecto |
|---|---|---|---|
| Mann-Whitney | Retraso: clima adverso | <0,001 | r = 0,108 (pequeño) |
| Mann-Whitney | Retraso: nieve aproximada | <0,001 | r = 0,105 (pequeño) |
| Spearman | Retraso vs precipitación | <0,001 | ρ = 0,137 (débil) |
| Spearman | Retraso vs viento | <0,001 | ρ = 0,069 (débil) |
| Kruskal-Wallis | Retraso: franja horaria | <0,001 | ε² = 0,0161 (pequeño) |
| Kruskal-Wallis | Retraso: aerolínea | <0,001 | ε² = 0,0065 (pequeño) |
| Kruskal-Wallis | Retraso: día de la semana | <0,001 | ε² = 0,0096 (pequeño) |
| Chi-cuadrado | Cancelación vs clima adverso | <0,001 | V = 0,101 |
| Chi-cuadrado | Cancelación vs aerolínea | <0,001 | V = 0,090 |

**Conclusión general.** Todas las asociaciones probadas son estadísticamente significativas, resultado esperable con más de un millón de observaciones. Los tamaños del efecto son consistentemente pequeños: el factor con mayor peso relativo es el clima (adverso o nevoso) sobre la cancelación y sobre el retraso, seguido de la franja horaria. Ningún factor aislado explica gran parte de la variabilidad del retraso, lo cual es coherente con que este depende de la interacción de múltiples causas (congestión de red, aerolínea, clima, hora del día) más que de una sola.